# Use Case — Portfolio Heat-Risk Screening

**Who this is for**  
REIT asset managers, private real-estate investors, and property operations leads who need to rank a portfolio by heat exposure for CapEx planning, insurance renewal, divestiture review, or tenant-comfort SLA enforcement.

**The scenario**  
You manage a 10-asset commercial portfolio in downtown San Jose. Insurance premiums are climbing, tenants on upper floors are complaining about cooling performance, and leadership has asked for a heat-risk ranking before the CapEx cycle closes. Walking a building per day is not an option — you need a desk-first screening that surfaces the exposed assets, tells you *why* they are exposed, and translates that into numbers the investment committee will act on.

This notebook combines **your portfolio data** with **FortyGuard layers** to answer four questions:

1. **Which properties are hottest at design hour?**  ← heatmap × portfolio
2. **Why are they hot?**  ← satellite segmentation on the top exposures
3. **What does tenant comfort look like through the day?**  ← environmental parameters
4. **What does it mean for operating cost and insurance?**  ← business-translation layer on top

> **Bring your own portfolio.** Sample data ships at `data/sample_real_estate_portfolio.csv`. Swap the path in Step 1 — as long as the columns match (`property_id`, `name`, `type`, `year_built`, `sqft`, `market_value_musd`, `latitude`, `longitude`), everything downstream works.

---

## Setup

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import pandas as pd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point, shape

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON

client = FortyGuardClient()

AOI              = SAN_JOSE_POLYGON      # ~104 km² (~40 mi²) across central San Jose
STUDY_DATE       = '2024-07-15'
STUDY_HOUR       = '14:00'
GRANULARITY_M    = 100                   # 100 m keeps the heatmap tile count tractable over this AOI
TOP_N_TO_ENRICH  = 3                     # expensive endpoints only run for this many hottest properties

print(f'Authenticated to {client.base_url}')

---
## Step 1 — Load your portfolio

### What you are doing
Reading the portfolio table. Any columns can ride along — the workflow only needs `latitude` and `longitude` to do the geospatial work; the rest are passed through to the final output so the investment committee sees the asset IDs and values they already recognize.

### Why this matters
This is the operations system of record. Starting here means the output carries *your* property IDs, *your* asset types, *your* square-footages. That is what lets finance and ops reuse the result without re-keying anything.

In [ ]:
portfolio = pd.read_csv(ROOT / 'data' / 'sample_real_estate_portfolio.csv')
print(f'Loaded {len(portfolio)} properties, total value $'
      f"{portfolio['market_value_musd'].sum():.0f} M, total {portfolio['sqft'].sum():,} sqft")
portfolio

---
## Step 2 — Generate the heat layer

### What you are doing
One heatmap covering the full portfolio AOI at the design-peak hour. One call, many properties — cheap.

### Why this matters
A city-wide weather observation misses intra-city variation that *actually* drives cooling cost and tenant complaints. At 80 m resolution you can distinguish a building on the hot side of a block from one on the cool side.

In [ ]:
heatmap = client.create_heatmap(
    polygon_aoi=AOI, start_date=STUDY_DATE, start_time=STUDY_HOUR,
    filter_type=1, granularity=GRANULARITY_M,
)
map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []
print(f'Returned {len(features)} tiles at {GRANULARITY_M} m resolution')

---
## Step 3 — Attach temperature to each property

### What you are doing
For each property, find the heatmap tile that contains its coordinates and copy that tile's temperature onto the row. Now every property has a `temperature_c` column — the analysis-ready table.

### Why this matters
This is the moment your portfolio table becomes a risk table. Every downstream question — ranking, scoring, business translation — is a `groupby` or a sort on this DataFrame.

In [ ]:
tile_polys = [(shape(f['geometry']), f['properties'].get('temperature')) for f in features]

def _temp_at(lat, lon):
    p = Point(lon, lat)
    for poly, t in tile_polys:
        if poly.contains(p):
            return t
    if not tile_polys: return None
    return min(tile_polys, key=lambda pt: pt[0].centroid.distance(p))[1]

portfolio['temperature_c'] = portfolio.apply(
    lambda r: _temp_at(r.latitude, r.longitude), axis=1)
portfolio = portfolio.sort_values('temperature_c', ascending=False).reset_index(drop=True)
portfolio.insert(0, 'temp_rank', portfolio.index + 1)
portfolio[['temp_rank', 'property_id', 'name', 'type', 'sqft', 'temperature_c']]

---
## Step 4 — Characterize the top exposures with satellite context

### What you are doing
For the top-3 hottest properties, call satellite segmentation. We extract the share of impervious surface (rooftops + roads) and vegetation in each property's surroundings.

### Why this matters
Temperature alone does not tell you what to spend money on. A hot property surrounded by impervious rooftops is a cool-roof / reflective-pavement candidate. A hot property with minimal vegetation is a planting candidate. The surface mix picks the retrofit.

In [ ]:
IMPERV = {'road', 'roads', 'pavement', 'building', 'buildings', 'rooftop', 'rooftops', 'bare'}
VEGGIE = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}

def _bucket(segments, keys):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

top = portfolio.head(TOP_N_TO_ENRICH).copy()
imps, vegs = [], []
for _, r in top.iterrows():
    print(f"  satellite: #{r.temp_rank} {r.property_id}")
    sat = client.satellite_segmentation(
        latitude=r.latitude, longitude=r.longitude,
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=1, granularity=GRANULARITY_M, verbose=False)
    segs = sat['result'].get('segmentation', {}).get('segments', {}) or {}
    imps.append(_bucket(segs, IMPERV)); vegs.append(_bucket(segs, VEGGIE))
top['impervious_pct'] = imps
top['vegetation_pct'] = vegs
top[['temp_rank', 'property_id', 'name', 'temperature_c', 'impervious_pct', 'vegetation_pct']]

---
## Step 5 — Profile tenant-comfort risk

### What you are doing
For each of the same top-3, pull environmental parameters across the design day and record the peak heat index.

### Why this matters
Tenant comfort SLAs are written in terms of heat index, not dry-bulb temperature. A building at 34 °C ambient with low humidity is comfortable; one at 31 °C with 80 % humidity is not. Heat index is the number complaints cluster around.

In [ ]:
peaks = []
for _, r in top.iterrows():
    print(f"  env_params: #{r.temp_rank} {r.property_id}")
    env = client.environmental_parameters(
        latitude=r.latitude, longitude=r.longitude,
        temperature=float(r.temperature_c),
        start_date=STUDY_DATE, start_time='09:00', end_time='18:00',
        filter_type=2, verbose=False)
    hi = (env['result']['locations'][0].get('parameters', {})
            .get('heat_index_celsius') or [])
    peaks.append(max(hi) if hi else None)
top['peak_heat_index_c'] = peaks
top[['temp_rank', 'property_id', 'name', 'temperature_c', 'peak_heat_index_c']]

---
## Step 6 — Composite heat-risk score

### What you are doing
Normalizing the three diagnostic factors (temperature, impervious fraction, peak heat index) to 0–1 across the analyzed subset, then combining them with weights:

- **50 %** tile temperature (the primary signal)
- **25 %** impervious surroundings (UHI amplifier)
- **25 %** peak heat index (tenant-experience amplifier)

### Why this matters
The investment committee wants one number per asset. The raw temperature is not defensible on its own — two buildings with identical ambient can have very different comfort and operating-cost profiles. The composite captures that.

In [ ]:
def _mm(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

top['temp_norm']   = _mm(top['temperature_c'])
top['imp_norm']    = _mm(top['impervious_pct'])
top['hi_norm']     = _mm(top['peak_heat_index_c'])
top['heat_risk_score'] = (0.50 * top['temp_norm']
                          + 0.25 * top['imp_norm']
                          + 0.25 * top['hi_norm']).round(3)
top[['property_id', 'name', 'temperature_c', 'impervious_pct',
     'peak_heat_index_c', 'heat_risk_score']]

---
## Step 7 — Business translation

### What you are doing
Converting the technical numbers into the three outputs the investment committee actually uses:

- **Cooling OpEx uplift (USD / year)** — extra kWh from running above a 24 °C baseline, priced at $0.24/kWh. Simple linear model: `(Δ°C above 24) × sqft × cooling_intensity × price`. Good enough for triage.
- **Insurance risk tier (A / B / C)** — percentile buckets on the composite score.
- **Tenant-comfort flag** — boolean, flips when peak heat index is above 32 °C.

### Why this matters
"Property X has a composite heat-risk score of 0.82" is a modeling statement. "Property X is in insurance tier C, carries ~$58 k/year in avoidable cooling OpEx, and will breach the tenant-comfort SLA on design-peak days" is a decision.

In [ ]:
BASELINE_C         = 24.0    # comfortable ambient baseline
COOLING_KWH_PER_SF = 0.18    # extra kWh per sqft per °C above baseline per season
KWH_PRICE_USD      = 0.24

def _opex_uplift(temp_c, sqft):
    if pd.isna(temp_c): return None
    delta = max(0.0, temp_c - BASELINE_C)
    return round(delta * sqft * COOLING_KWH_PER_SF * KWH_PRICE_USD, 0)

def _tier(score):
    if pd.isna(score): return 'unranked'
    if score >= 0.67:  return 'C'
    if score >= 0.33:  return 'B'
    return 'A'

def _comfort_flag(hi):
    if pd.isna(hi): return None
    return bool(hi > 32.0)

# Apply to the enriched top; the rest of the portfolio keeps base columns only.
portfolio = portfolio.merge(
    top[['property_id', 'impervious_pct', 'vegetation_pct',
         'peak_heat_index_c', 'heat_risk_score']],
    on='property_id', how='left',
)
portfolio['cooling_opex_uplift_usd'] = portfolio.apply(
    lambda r: _opex_uplift(r.temperature_c, r.sqft), axis=1)
portfolio['insurance_tier']   = portfolio['heat_risk_score'].apply(_tier)
portfolio['comfort_sla_risk'] = portfolio['peak_heat_index_c'].apply(_comfort_flag)

cols = ['temp_rank', 'property_id', 'name', 'type', 'sqft',
        'temperature_c', 'impervious_pct', 'peak_heat_index_c',
        'heat_risk_score', 'insurance_tier',
        'cooling_opex_uplift_usd', 'comfort_sla_risk']
portfolio[cols]

In [ ]:
out = ROOT / 'outputs' / 'portfolio_heat_risk.csv'
out.parent.mkdir(parents=True, exist_ok=True)
portfolio[cols].to_csv(out, index=False)
print(f'Saved portfolio risk table to {out}')
print(f"Aggregate avoidable cooling OpEx: $"
      f"{portfolio['cooling_opex_uplift_usd'].sum():,.0f} / year")

---
## Wrap-up

Starting from a portfolio CSV you now have:

| Artifact | Audience |
|----------|----------|
| Temperature-joined portfolio table | Asset team |
| Top-N composite risk score | Investment committee |
| Annualized cooling OpEx uplift per asset | Finance / budgeting |
| Insurance risk tier | Risk management / broker |
| Tenant-comfort SLA flag | Property operations / leasing |

Every number traces back to a measurement, not an assumption. That is what makes this output defensible at the quarterly review — and what lets you re-run it next year to see whether last year's retrofit moved the needle.

**Apply this pattern to adjacent use cases**: swap the CSV for insured-properties portfolio (insurance underwriting), data-center sites (operational-risk screening), or hospitality assets (guest-comfort benchmarking). The workflow — portfolio × heatmap × enrichment → risk-scored table — is the same.